# QCentroid - Railway Rolling Stock MWIS Solver

**Quantum solver for Maximum Weighted Independent Set (MWIS) using IQM Resonance**

This notebook demonstrates the QCentroid algorithm for railway rolling-stock cycle selection using QAOA on IQM's quantum hardware.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '.')

import json
from qcentroid import run

print("✓ QCentroid module imported successfully")

## 2. Define Test Data

In [ ]:
# Example input: Railway rolling-stock cycles with conflicts
example_input = {
    "nodes": [
        {
            "id": "C01",
            "weight": 85.0,
            "trips": ["T01", "T02"],
            "passenger_km": 8000,
            "empty_km": 100,
            "operating_cost": 1200,
        },
        {
            "id": "C02",
            "weight": 70.0,
            "trips": ["T02", "T03"],
            "passenger_km": 7000,
            "empty_km": 250,
            "operating_cost": 1250,
        },
        {
            "id": "C03",
            "weight": 90.0,
            "trips": ["T04", "T05"],
            "passenger_km": 9000,
            "empty_km": 50,
            "operating_cost": 1180,
        },
        {
            "id": "C04",
            "weight": 60.0,
            "trips": ["T01", "T06"],
            "passenger_km": 6000,
            "empty_km": 300,
            "operating_cost": 1300,
        },
        {
            "id": "C05",
            "weight": 75.0,
            "trips": ["T05", "T07"],
            "passenger_km": 7500,
            "empty_km": 80,
            "operating_cost": 1190,
        },
    ],
    "edges": [
        ["C01", "C02"],
        ["C01", "C04"],
        ["C03", "C05"],
    ],
}

print("Test Data Summary:")
print(f"  Nodes (cycles): {len(example_input['nodes'])}")
print(f"  Edges (conflicts): {len(example_input['edges'])}")
print(f"  Cycles: {[node['id'] for node in example_input['nodes']]}")
print(f"  Conflicts: {example_input['edges']}")

## 3. Configure Solver Parameters

**Important:** Set your IQM token below to use real quantum hardware. Without a token, the solver will fall back to classical simulation.

In [ ]:
# IQM Configuration
# TODO: Replace with your actual IQM Resonance token
IQM_TOKEN = None  # Set to your token to use real quantum hardware
QUANTUM_COMPUTER = "emerald"  # Options: emerald, sirius, garnet, sapphire, ruby, diamond
IQM_SERVER_URL = "https://resonance.iqm.tech/"

# Solver parameters
example_params = {
    "iqm_token": IQM_TOKEN,
    "quantum_computer": QUANTUM_COMPUTER,
    "server_url": IQM_SERVER_URL,
    "shots": 512,
    "qaoa_depth": 1,
}

print("Solver Configuration:")
print(f"  Backend: {'IQM Resonance (requires token)' if IQM_TOKEN else 'Classical Greedy / Qiskit AER'}")
print(f"  Quantum Computer: {QUANTUM_COMPUTER}")
print(f"  Shots: {example_params['shots']}")
print(f"  QAOA Depth: {example_params['qaoa_depth']}")

## 4. Execute the Solver

In [ ]:
# Run the solver
result = run(example_input, example_params, {})

print("\n" + "="*80)
print("SOLVER EXECUTION COMPLETED")
print("="*80)

## 5. Results and Metrics

In [ ]:
# Display main results
print("\n📊 SOLUTION SUMMARY")
print("-" * 80)
print(f"Selected Cycles: {result['selected_cycles']}")
print(f"Total Weight: {result['total_weight']:.4f}")
print(f"Is Feasible: {result['is_feasible']}")
print(f"Backend Used: {result['backend_used']}")

if 'error' in result:
    print(f"❌ Error: {result['error']}")

In [ ]:
# Display trip coverage metrics
print("\n🚂 TRIP COVERAGE ANALYSIS")
print("-" * 80)
print(f"Scheduled Trips: {result['scheduled_trips']}")
print(f"Covered Trips: {result['covered_trips']}")
if result['coverage_rate'] is not None:
    print(f"Coverage Rate: {result['coverage_percent']:.2f}%")
else:
    print("Coverage Rate: N/A (no trip data available)")

In [ ]:
# Display operational metrics
print("\n⚙️  OPERATIONAL METRICS")
print("-" * 80)
if result['total_empty_km'] is not None:
    print(f"Total Empty-KM: {result['total_empty_km']:.2f} km")
else:
    print("Total Empty-KM: N/A")

if result['total_passenger_km'] is not None:
    print(f"Total Passenger-KM: {result['total_passenger_km']:.2f} km")
else:
    print("Total Passenger-KM: N/A")

if result['total_operating_cost'] is not None:
    print(f"Total Operating Cost: €{result['total_operating_cost']:.2f}")
else:
    print("Total Operating Cost: N/A")

In [ ]:
# Display execution metrics
print("\n⏱️  EXECUTION METRICS")
print("-" * 80)
metrics = result['execution_metrics']
print(f"Execution Time: {metrics['execution_time_seconds']:.4f} seconds")
print(f"Initial Solution Size: {metrics['initial_solution_size']}")
print(f"Final Solution Size: {metrics['final_solution_size']}")
print(f"QAOA Depth: {metrics['qaoa_depth']}")
print(f"Shots: {metrics['shots']}")
print(f"Penalty Parameter: {metrics['penalty']}")

if metrics['training_cost'] is not None:
    print(f"Training Cost: {metrics['training_cost']:.6f}")

if metrics['job_id']:
    print(f"Job ID: {metrics['job_id']}")

In [ ]:
# Display pruning statistics
print("\n🔧 PRUNING STATISTICS")
print("-" * 80)
pruning = result['pruning_stats']
print(f"Nodes Pruned: {pruning['nodes_pruned']}")
print(f"Pruned Cycles: {pruning['pruned_nodes']}")

## 6. Full Result JSON

In [ ]:
# Display full result as formatted JSON
print(json.dumps(result, indent=2, default=str))

## 7. Visualization (if available)

If a conflict graph visualization was generated, load and display it here.

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

# Check if conflict graph visualization exists
if 'assets' in result and 'conflict_graph_png' in result['assets']:
    graph_path = result['assets']['conflict_graph_png']
    
    if os.path.exists(graph_path):
        print(f"Loading visualization from: {graph_path}")
        img = Image.open(graph_path)
        
        fig, ax = plt.subplots(figsize=(16, 12))
        ax.imshow(img)
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        print("✓ Conflict graph visualization displayed")
    else:
        print(f"⚠️  Visualization file not found: {graph_path}")
else:
    print("⚠️  No conflict graph visualization available")

## 8. How to Use with IQM Token

To run this notebook with actual IQM Resonance hardware:

1. **Obtain an IQM Token:** Get your token from the IQM Resonance portal
2. **Update Cell 3:** Replace `IQM_TOKEN = None` with your actual token
3. **Choose Quantum Computer:** Select from available QPUs (emerald, sirius, etc.)
4. **Adjust Parameters:**
   - `shots`: Number of measurement repetitions (higher = more reliable)
   - `qaoa_depth`: QAOA circuit depth (higher = potentially better solutions, but longer runtime)
5. **Re-run Cell 4** to execute on IQM hardware

**Note:** Classical optimization of QAOA parameters occurs locally, then the optimized circuit is sent to IQM for execution.